In [0]:
from pathlib import Path
import json
import random

BASE_PATH = Path("/Volumes/de_prac/day03/data-files")

INCOMING_DIR = BASE_PATH / "incoming"
BRONZE_DIR = BASE_PATH / "bronze"
SILVER_DIR = BASE_PATH / "silver"
QUARANTINE_DIR = BASE_PATH / "quarantine"
REPORTS_DIR = BASE_PATH / "reports"
LOGS_DIR = BASE_PATH / "logs"

SOURCE_FILE = INCOMING_DIR / "orders.jsonl"

print("Source file:", SOURCE_FILE)

In [0]:
random.seed(42)

TOTAL_LINES = 100_000


def create_order(order_number):
    order_id = f"ORD{order_number:06d}"
    customer_id = f"CUST{random.randint(1, 50000):06d}"

    item_count = random.randint(1, 4)

    items = []

    for _ in range(item_count):
        items.append(
            {
                "product_id": f"PROD{random.randint(1, 5000):05d}",
                "quantity": random.randint(1, 5),
                "price": round(random.uniform(5, 500), 2),
            }
        )

    return {
        "order_id": order_id,
        "customer": {
            "customer_id": customer_id,
            "name": f"Customer {customer_id}",
        },
        "items": items,
        "shipping": {
            "address": f"{random.randint(100, 9999)} Main Street",
            "city": random.choice(
                ["Columbus", "Austin", "New York", "Chicago", "Seattle"]
            ),
            "state": random.choice(
                ["OH", "TX", "NY", "IL", "WA"]
            ),
            "country": "USA",
        },
    }

In [0]:
with open(SOURCE_FILE, "w", encoding="utf-8") as f:

    # Line 1: valid order with TWO items
    order_1 = {
        "order_id": "ORD000001",
        "customer": {
            "customer_id": "CUST000001",
            "name": "Customer One",
        },
        "items": [
            {
                "product_id": "PROD00001",
                "quantity": 2,
                "price": 25.50,
            },
            {
                "product_id": "PROD00002",
                "quantity": 1,
                "price": 40.00,
            },
        ],
        "shipping": {
            "address": "101 Main Street",
            "city": "Columbus",
            "state": "OH",
            "country": "USA",
        },
    }

    f.write(json.dumps(order_1) + "\n")

    # Line 2: valid order with ONE item
    order_2 = {
        "order_id": "ORD000002",
        "customer": {
            "customer_id": "CUST000002",
            "name": "Customer Two",
        },
        "items": [
            {
                "product_id": "PROD00003",
                "quantity": 3,
                "price": 15.75,
            }
        ],
        "shipping": {
            "address": "202 Main Street",
            "city": "Austin",
            "state": "TX",
            "country": "USA",
        },
    }

    f.write(json.dumps(order_2) + "\n")

    # Line 3: valid order with EMPTY item array
    order_3 = {
        "order_id": "ORD000003",
        "customer": {
            "customer_id": "CUST000003",
            "name": "Customer Three",
        },
        "items": [],
        "shipping": {
            "address": "303 Main Street",
            "city": "Chicago",
            "state": "IL",
            "country": "USA",
        },
    }

    f.write(json.dumps(order_3) + "\n")

    # Line 4: invalid business record - customer missing
    order_4 = {
        "order_id": "ORD000004",
        "items": [
            {
                "product_id": "PROD00004",
                "quantity": 1,
                "price": 99.99,
            }
        ],
        "shipping": {
            "address": "404 Main Street",
            "city": "Seattle",
            "state": "WA",
            "country": "USA",
        },
    }

    f.write(json.dumps(order_4) + "\n")

    # Line 5: deliberately MALFORMED JSON
    f.write(
        '{"order_id":"ORD000005",'
        '"customer":{"customer_id":"CUST000005"},'
        '"items":[{"product_id":"PROD00005","quantity":2,"price":10.50}]'
        "\n"
    )

    # Lines 6 - 100,000
    for order_number in range(6, TOTAL_LINES + 1):
        order = create_order(order_number)
        f.write(json.dumps(order) + "\n")


print(f"Created {SOURCE_FILE}")
print(f"Expected source lines: {TOTAL_LINES:,}")

In [0]:
line_count = 0

with open(SOURCE_FILE, "r", encoding="utf-8") as f:
    for _ in f:
        line_count += 1

print(f"Actual source lines: {line_count:,}")

In [0]:
with open(SOURCE_FILE, "r", encoding="utf-8") as f:
    for line_number in range(1, 6):
        line = f.readline()
        print(f"LINE {line_number}")
        print(line[:500])
        print("-" * 100)

saving the file into bronze


In [0]:
import shutil
from pathlib import Path

BRONZE_FILE = BRONZE_DIR / "orders.jsonl"

shutil.copyfile(SOURCE_FILE, BRONZE_FILE)

print("Incoming file:", SOURCE_FILE)
print("Bronze copy:  ", BRONZE_FILE)
print("Bronze raw file preserved successfully.")

In [0]:
from datetime import datetime, timezone
from pyspark.sql.types import (
    StructType,
    StructField,
    LongType,
    StringType,
    TimestampType,
)

bronze_rows = []

ingestion_time = datetime.now(timezone.utc)

with open(BRONZE_FILE, "r", encoding="utf-8") as f:
    for line_number, line in enumerate(f, start=1):
        bronze_rows.append(
            (
                line_number,
                line.rstrip("\n"),
                str(BRONZE_FILE),
                ingestion_time,
            )
        )

bronze_schema = StructType(
    [
        StructField("line_number", LongType(), False),
        StructField("raw_record", StringType(), False),
        StructField("source_file", StringType(), False),
        StructField("ingestion_timestamp", TimestampType(), False),
    ]
)

bronze_df = spark.createDataFrame(
    bronze_rows,
    schema=bronze_schema
)

display(bronze_df.limit(10))

In [0]:
BRONZE_TABLE = "de_prac.day03.orders_bronze"

(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(BRONZE_TABLE)
)

print(f"Created table: {BRONZE_TABLE}")

In [0]:
bronze_count = spark.table(BRONZE_TABLE).count()

print(f"Bronze row count: {bronze_count:,}")
print(f"Expected count:   {TOTAL_LINES:,}")

assert bronze_count == TOTAL_LINES

print("BRONZE COUNT CHECK: PASS")

In [0]:
display(
    spark.table(BRONZE_TABLE)
    .filter("line_number <= 5")
    .orderBy("line_number")
)

## Step 3 - Parse JSON and Identify Invalid Records

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    ArrayType,
    IntegerType,
    DoubleType,
)

order_schema = StructType([
    StructField("order_id", StringType(), True),

    StructField(
        "customer",
        StructType([
            StructField("customer_id", StringType(), True),
            StructField("name", StringType(), True),
        ]),
        True,
    ),

    StructField(
        "items",
        ArrayType(
            StructType([
                StructField("product_id", StringType(), True),
                StructField("quantity", IntegerType(), True),
                StructField("price", DoubleType(), True),
            ])
        ),
        True,
    ),

    StructField(
        "shipping",
        StructType([
            StructField("address", StringType(), True),
            StructField("city", StringType(), True),
            StructField("state", StringType(), True),
            StructField("country", StringType(), True),
        ]),
        True,
    ),

    # Spark places malformed JSON here
    StructField("_corrupt_record", StringType(), True),
])

In [0]:
bronze_df = spark.table("de_prac.day03.orders_bronze")

parsed_df = (
    bronze_df
    .withColumn(
        "parsed",
        F.from_json(
            F.col("raw_record"),
            order_schema,
            {
                "mode": "PERMISSIVE",
                "columnNameOfCorruptRecord": "_corrupt_record",
            },
        ),
    )
)

display(
    parsed_df
    .filter(F.col("line_number") <= 5)
    .select(
        "line_number",
        "raw_record",
        "parsed.order_id",
        "parsed.customer",
        "parsed.items",
        "parsed._corrupt_record",
    )
    .orderBy("line_number")
)

In [0]:
malformed_df = (
    parsed_df
    .filter(F.col("parsed._corrupt_record").isNotNull())
    .select(
        "line_number",
        "raw_record",
        "source_file",
        "ingestion_timestamp",
    )
    .withColumn("reject_reason", F.lit("MALFORMED_JSON"))
)

display(malformed_df)

In [0]:
malformed_count = malformed_df.count()

print(f"Malformed JSON records: {malformed_count}")

assert malformed_count == 1

print("MALFORMED JSON CHECK: PASS")

In [0]:
valid_json_df = (
    parsed_df
    .filter(F.col("parsed._corrupt_record").isNull())
    .select(
        "line_number",
        "raw_record",
        F.col("parsed.order_id").alias("order_id"),
        F.col("parsed.customer").alias("customer"),
        F.col("parsed.items").alias("items"),
        F.col("parsed.shipping").alias("shipping"),
        "source_file",
        "ingestion_timestamp",
    )
)

In [0]:
valid_json_count = valid_json_df.count()

print(f"Valid JSON lines: {valid_json_count:,}")
print(f"Malformed lines:  {malformed_count:,}")
print(f"Total:            {valid_json_count + malformed_count:,}")

assert valid_json_count + malformed_count == TOTAL_LINES

print("JSON PARSING RECONCILIATION: PASS")

## Step 4 - Validate Orders and Create Header Audit

In [0]:
from pyspark.sql import functions as F

validated_df = (
    valid_json_df
    .withColumn(
        "validation_error",
        F.when(
            F.col("order_id").isNull() | (F.trim(F.col("order_id")) == ""),
            F.lit("MISSING_ORDER_ID"),
        )
        .when(
            F.col("customer").isNull(),
            F.lit("MISSING_CUSTOMER"),
        )
        .when(
            F.col("customer.customer_id").isNull()
            | (F.trim(F.col("customer.customer_id")) == ""),
            F.lit("MISSING_CUSTOMER_ID"),
        )
        .when(
            F.col("items").isNull(),
            F.lit("MISSING_ITEMS_ARRAY"),
        )
    )
)

In [0]:
business_rejects_df = (
    validated_df
    .filter(F.col("validation_error").isNotNull())
    .select(
        "line_number",
        "raw_record",
        "source_file",
        "ingestion_timestamp",
        F.col("validation_error").alias("reject_reason"),
    )
)

display(business_rejects_df)

In [0]:
business_reject_count = business_rejects_df.count()

print(f"Business validation rejects: {business_reject_count}")

assert business_reject_count == 1

print("BUSINESS VALIDATION CHECK: PASS")

In [0]:
rejected_df = (
    malformed_df
    .unionByName(business_rejects_df)
    .orderBy("line_number")
)

display(rejected_df)

In [0]:
rejected_count = rejected_df.count()

print(f"Total rejected records: {rejected_count}")

assert rejected_count == 2

print("REJECT COUNT CHECK: PASS")

In [0]:
REJECTED_TABLE = "de_prac.day03.orders_rejected"

(
    rejected_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(REJECTED_TABLE)
)

print(f"Created table: {REJECTED_TABLE}")

In [0]:
accepted_orders_df = (
    validated_df
    .filter(F.col("validation_error").isNull())
    .select(
        "line_number",
        "order_id",
        F.col("customer.customer_id").alias("customer_id"),
        F.col("customer.name").alias("customer_name"),
        "items",
        F.col("shipping.address").alias("shipping_address"),
        F.col("shipping.city").alias("shipping_city"),
        F.col("shipping.state").alias("shipping_state"),
        F.col("shipping.country").alias("shipping_country"),
        "source_file",
        "ingestion_timestamp",
    )
    .withColumn(
        "expected_item_count",
        F.size(F.col("items"))
    )
)

In [0]:
accepted_count = accepted_orders_df.count()

print(f"Accepted order headers: {accepted_count:,}")

In [0]:
order_header_audit_df = (
    accepted_orders_df
    .select(
        "line_number",
        "order_id",
        "customer_id",
        "customer_name",
        "expected_item_count",
        "shipping_address",
        "shipping_city",
        "shipping_state",
        "shipping_country",
        "source_file",
        "ingestion_timestamp",
    )
)

In [0]:
HEADER_TABLE = "de_prac.day03.order_header_audit"

(
    order_header_audit_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(HEADER_TABLE)
)

print(f"Created table: {HEADER_TABLE}")

In [0]:
display(
    order_header_audit_df
    .filter(F.col("order_id") == "ORD000003")
)

In [0]:
source_count = spark.table(BRONZE_TABLE).count()
accepted_count = order_header_audit_df.count()
rejected_count = rejected_df.count()

print(f"Source lines:      {source_count:,}")
print(f"Accepted headers: {accepted_count:,}")
print(f"Rejected lines:   {rejected_count:,}")
print(f"Reconciled total: {accepted_count + rejected_count:,}")

assert source_count == accepted_count + rejected_count

print("SOURCE-TO-HEADER RECONCILIATION: PASS")

## Step 5 - Flatten Order Items

In [0]:
from pyspark.sql import functions as F

flattened_items_df = (
    accepted_orders_df
    .select(
        "line_number",
        "order_id",
        "customer_id",
        F.posexplode(F.col("items")).alias("item_index", "item"),
        "shipping_address",
        "shipping_city",
        "shipping_state",
        "shipping_country",
        "source_file",
        "ingestion_timestamp",
    )
    .select(
        "line_number",
        "order_id",
        "customer_id",
        "item_index",
        F.col("item.product_id").alias("product_id"),
        F.col("item.quantity").alias("quantity"),
        F.col("item.price").alias("price"),
        "shipping_address",
        "shipping_city",
        "shipping_state",
        "shipping_country",
        "source_file",
        "ingestion_timestamp",
    )
)

display(flattened_items_df.limit(20))

In [0]:
F.posexplode(F.col("items")).alias("item_index", "item")

In [0]:
display(
    flattened_items_df
    .filter(
        F.col("order_id").isin(
            "ORD000001",
            "ORD000002",
            "ORD000003"
        )
    )
    .orderBy("order_id", "item_index")
)

In [0]:
ITEM_TABLE = "de_prac.day03.orders_items_flattened"

(
    flattened_items_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(ITEM_TABLE)
)

print(f"Created table: {ITEM_TABLE}")

In [0]:
expected_item_rows = (
    accepted_orders_df
    .agg(
        F.sum(F.size(F.col("items"))).alias("expected_items")
    )
    .first()["expected_items"]
)

actual_item_rows = flattened_items_df.count()

print(f"Expected item rows: {expected_item_rows:,}")
print(f"Actual item rows:   {actual_item_rows:,}")

In [0]:
assert expected_item_rows == actual_item_rows

print("ITEM COUNT RECONCILIATION: PASS")

In [0]:
order_1_count = (
    flattened_items_df
    .filter(F.col("order_id") == "ORD000001")
    .count()
)

order_2_count = (
    flattened_items_df
    .filter(F.col("order_id") == "ORD000002")
    .count()
)

order_3_count = (
    flattened_items_df
    .filter(F.col("order_id") == "ORD000003")
    .count()
)

print("ORD000001 item rows:", order_1_count)
print("ORD000002 item rows:", order_2_count)
print("ORD000003 item rows:", order_3_count)

assert order_1_count == 2
assert order_2_count == 1
assert order_3_count == 0

print("EDGE CASE ITEM CHECKS: PASS")


## Step 6 - Final Reconciliation and Audit Report

In [0]:
from pyspark.sql import functions as F

source_count = spark.table("de_prac.day03.orders_bronze").count()
accepted_header_count = spark.table("de_prac.day03.order_header_audit").count()
rejected_count = spark.table("de_prac.day03.orders_rejected").count()
item_output_count = spark.table("de_prac.day03.orders_items_flattened").count()

malformed_count = (
    spark.table("de_prac.day03.orders_rejected")
    .filter(F.col("reject_reason") == "MALFORMED_JSON")
    .count()
)

business_reject_count = (
    spark.table("de_prac.day03.orders_rejected")
    .filter(F.col("reject_reason") != "MALFORMED_JSON")
    .count()
)

expected_item_count = (
    accepted_orders_df
    .agg(F.sum(F.size("items")).alias("expected_items"))
    .first()["expected_items"]
)

In [0]:
source_recon_status = (
    "PASS"
    if source_count == accepted_header_count + rejected_count
    else "FAIL"
)

item_recon_status = (
    "PASS"
    if expected_item_count == item_output_count
    else "FAIL"
)

print("DAY 3 RECONCILIATION")
print("-" * 50)

print(f"Source lines:              {source_count:,}")
print(f"Accepted order headers:   {accepted_header_count:,}")
print(f"Rejected lines:           {rejected_count:,}")
print(f"  Malformed JSON:         {malformed_count:,}")
print(f"  Business rejects:       {business_reject_count:,}")
print()

print(
    f"Source reconciliation: "
    f"{accepted_header_count + rejected_count:,} / {source_count:,} "
    f"-> {source_recon_status}"
)

print()

print(f"Expected item rows:       {expected_item_count:,}")
print(f"Actual flattened rows:    {item_output_count:,}")
print(f"Item reconciliation:      {item_recon_status}")

In [0]:
assert source_recon_status == "PASS"
assert item_recon_status == "PASS"

print("FINAL DAY 3 RECONCILIATION: PASS")

In [0]:
reconciliation_data = [
    (
        "SOURCE_LINE_RECONCILIATION",
        source_count,
        accepted_header_count + rejected_count,
        source_recon_status
    ),
    (
        "ITEM_ROW_RECONCILIATION",
        expected_item_count,
        item_output_count,
        item_recon_status
    ),
]

reconciliation_df = spark.createDataFrame(
    reconciliation_data,
    [
        "check_name",
        "expected_count",
        "actual_count",
        "status"
    ]
)

display(reconciliation_df)

In [0]:
RECON_TABLE = "de_prac.day03.reconciliation_summary"

(
    reconciliation_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(RECON_TABLE)
)

print(f"Created table: {RECON_TABLE}")

In [0]:
import json
from datetime import datetime, timezone

REPORT_FILE = (
    "/Volumes/de_prac/day03/data-files/"
    "reports/day03_reconciliation.json"
)

report = {
    "run_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "source_lines": source_count,
    "accepted_order_headers": accepted_header_count,
    "rejected_lines": rejected_count,
    "malformed_json_records": malformed_count,
    "business_validation_rejects": business_reject_count,
    "expected_item_rows": expected_item_count,
    "actual_item_rows": item_output_count,
    "source_reconciliation_status": source_recon_status,
    "item_reconciliation_status": item_recon_status,
}

with open(REPORT_FILE, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print(f"Report created: {REPORT_FILE}")

In [0]:
with open(REPORT_FILE, "r", encoding="utf-8") as f:
    print(f.read())

## Step 7 - Automated Tests

In [0]:
import unittest
from pyspark.sql import functions as F


class Day03OrdersTests(unittest.TestCase):

    @classmethod
    def setUpClass(cls):
        cls.bronze = spark.table("de_prac.day03.orders_bronze")
        cls.headers = spark.table("de_prac.day03.order_header_audit")
        cls.items = spark.table("de_prac.day03.orders_items_flattened")
        cls.rejects = spark.table("de_prac.day03.orders_rejected")
        cls.recon = spark.table("de_prac.day03.reconciliation_summary")

    # --------------------------------------------------
    # Source / Bronze tests
    # --------------------------------------------------

    def test_01_bronze_count(self):
        self.assertEqual(
            self.bronze.count(),
            100_000
        )

    def test_02_line_numbers_preserved(self):
        result = (
            self.bronze
            .agg(
                F.min("line_number").alias("min_line"),
                F.max("line_number").alias("max_line")
            )
            .first()
        )

        self.assertEqual(result["min_line"], 1)
        self.assertEqual(result["max_line"], 100_000)

    # --------------------------------------------------
    # Required fixture / edge-case tests
    # --------------------------------------------------

    def test_03_two_item_order(self):
        count = (
            self.items
            .filter(F.col("order_id") == "ORD000001")
            .count()
        )

        self.assertEqual(count, 2)

    def test_04_one_item_order(self):
        count = (
            self.items
            .filter(F.col("order_id") == "ORD000002")
            .count()
        )

        self.assertEqual(count, 1)

    def test_05_empty_item_order_header_preserved(self):
        header = (
            self.headers
            .filter(F.col("order_id") == "ORD000003")
            .first()
        )

        self.assertIsNotNone(header)
        self.assertEqual(header["expected_item_count"], 0)

    def test_06_empty_item_order_produces_zero_items(self):
        count = (
            self.items
            .filter(F.col("order_id") == "ORD000003")
            .count()
        )

        self.assertEqual(count, 0)

    def test_07_missing_customer_rejected(self):
        rows = (
            self.rejects
            .filter(F.col("line_number") == 4)
            .select("reject_reason")
            .collect()
        )

        self.assertEqual(len(rows), 1)
        self.assertEqual(
            rows[0]["reject_reason"],
            "MISSING_CUSTOMER"
        )

    def test_08_malformed_json_rejected(self):
        rows = (
            self.rejects
            .filter(F.col("line_number") == 5)
            .select("reject_reason")
            .collect()
        )

        self.assertEqual(len(rows), 1)
        self.assertEqual(
            rows[0]["reject_reason"],
            "MALFORMED_JSON"
        )

    # --------------------------------------------------
    # Traceability tests
    # --------------------------------------------------

    def test_09_order_id_retained_in_item_rows(self):
        missing_order_ids = (
            self.items
            .filter(
                F.col("order_id").isNull()
                | (F.trim("order_id") == "")
            )
            .count()
        )

        self.assertEqual(missing_order_ids, 0)

    def test_10_customer_id_retained(self):
        missing_customer_ids = (
            self.items
            .filter(
                F.col("customer_id").isNull()
                | (F.trim("customer_id") == "")
            )
            .count()
        )

        self.assertEqual(missing_customer_ids, 0)

    def test_11_item_index_preserved(self):
        rows = (
            self.items
            .filter(F.col("order_id") == "ORD000001")
            .orderBy("item_index")
            .select("item_index")
            .collect()
        )

        indexes = [row["item_index"] for row in rows]

        self.assertEqual(indexes, [0, 1])

    # --------------------------------------------------
    # Reconciliation tests
    # --------------------------------------------------

    def test_12_source_reconciliation(self):
        source_count = self.bronze.count()
        accepted_count = self.headers.count()
        rejected_count = self.rejects.count()

        self.assertEqual(
            source_count,
            accepted_count + rejected_count
        )

    def test_13_item_reconciliation(self):
        result = (
            self.recon
            .filter(
                F.col("check_name")
                == "ITEM_ROW_RECONCILIATION"
            )
            .first()
        )

        self.assertEqual(
            result["expected_count"],
            result["actual_count"]
        )

        self.assertEqual(
            result["status"],
            "PASS"
        )

    def test_14_source_reconciliation_status(self):
        result = (
            self.recon
            .filter(
                F.col("check_name")
                == "SOURCE_LINE_RECONCILIATION"
            )
            .first()
        )

        self.assertEqual(result["status"], "PASS")

In [0]:
suite = unittest.TestLoader().loadTestsFromTestCase(
    Day03OrdersTests
)

runner = unittest.TextTestRunner(
    verbosity=2
)

test_result = runner.run(suite)

In [0]:
tests_run = test_result.testsRun
failures = len(test_result.failures)
errors = len(test_result.errors)

print("DAY 3 TEST SUMMARY")
print("-" * 40)
print(f"Tests run: {tests_run}")
print(f"Failures:  {failures}")
print(f"Errors:    {errors}")

if failures == 0 and errors == 0:
    print("DAY 3 TEST SUITE: PASS")
else:
    print("DAY 3 TEST SUITE: FAIL")